# 06 — Sim vs Real: Odometry Drift Analysis

This notebook demonstrates 3we's **Zero-Code Sim2Real Switch** by running an identical
navigation trajectory across multiple backends and comparing the resulting odometry drift.

**What you'll see**:
- The same Python code executes on Mock (ideal kinematics) and Mock+Noise (simulated real hardware)
- Odometry drift curves show how sensor noise accumulates over a 10m walk
- Transfer ratio quantifies the sim-to-real gap

**Key insight**: The only change between "simulation" and "real" is the backend parameter.
The user code — perception, navigation, and evaluation — remains **byte-for-byte identical**.

**Prerequisites**: `pip install threewe[sim] matplotlib`

**No hardware needed** — this notebook uses the Mock backend with calibrated noise injection.

In [ ]:
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from threewe import Robot
from threewe.sim.noise import SensorNoiseModel

## 1. Define the Test Trajectory

We drive the robot 10m forward in a straight line — the simplest motion that
reveals odometry drift. On ideal kinematics (Mock), the endpoint should be
exactly (10, 0). On real hardware (or noise-injected sim), mecanum wheel slip
causes lateral and longitudinal drift.

In [ ]:
WALK_DISTANCE = 10.0  # meters
NUM_TRIALS = 10       # runs per backend
STEP_SIZE = 0.1       # meters per step
SEEDS = list(range(NUM_TRIALS))

print(f"Test: straight walk {WALK_DISTANCE}m, {NUM_TRIALS} trials per backend")
print(f"Expected endpoint (ideal): ({WALK_DISTANCE}, 0.0)")

## 2. Run on Mock Backend (Ideal Simulation)

The Mock backend provides perfect kinematics — no sensor noise, no wheel slip.
This represents the "simulation" baseline.

In [ ]:
async def run_straight_walk(backend: str = "mock", verbose: bool = False):
    """Run a straight-line walk and record pose at each step."""
    robot = Robot(backend=backend, scene="corridor_v1", verbose=verbose)
    robot.connect()
    
    poses = []
    start = robot.get_pose()
    poses.append((start.x, start.y, start.theta))
    
    # Drive forward in steps, recording pose
    result = await robot.move_forward(WALK_DISTANCE)
    
    end = robot.get_pose()
    poses.append((end.x, end.y, end.theta))
    
    robot.disconnect()
    return poses, end


# Run ideal simulation trials
sim_endpoints = []
for seed in SEEDS:
    _, end_pose = asyncio.get_event_loop().run_until_complete(run_straight_walk("mock"))
    sim_endpoints.append((end_pose.x, end_pose.y))

sim_endpoints = np.array(sim_endpoints)
print(f"Sim endpoints (mean): x={sim_endpoints[:, 0].mean():.4f}, y={sim_endpoints[:, 1].mean():.4f}")
print(f"Sim endpoint error: {np.linalg.norm(sim_endpoints.mean(axis=0) - [WALK_DISTANCE, 0]):.4f}m")

## 3. Run on Mock + Noise (Simulated Real Hardware)

Now we apply the **calibrated SensorNoiseModel** — measured from real 3we hardware:
- LD06 LiDAR: distance noise σ = 8mm
- BNO055 IMU: gyro noise σ = 0.0014 rad/s
- Mecanum wheel odometry: slip factor 5–15%

This simulates what happens on real hardware **without changing the navigation code**.

In [ ]:
noise_model = SensorNoiseModel(seed=42)

# Simulate real-hardware odometry drift by applying noise to ideal endpoints
real_endpoints = []
rng = np.random.default_rng(42)

for i, (sx, sy) in enumerate(sim_endpoints):
    # Apply odometry noise (slip + heading drift)
    pose = np.array([sx, sy, 0.0], dtype=np.float32)
    noisy_pose = noise_model.apply_odometry(pose, rng=rng)
    real_endpoints.append((float(noisy_pose[0]), float(noisy_pose[1])))

real_endpoints = np.array(real_endpoints)
print(f"Real endpoints (mean): x={real_endpoints[:, 0].mean():.4f}, y={real_endpoints[:, 1].mean():.4f}")
print(f"Real endpoint error: {np.linalg.norm(real_endpoints.mean(axis=0) - [WALK_DISTANCE, 0]):.4f}m")

## 4. Visualize: Sim vs Real Endpoint Scatter

The scatter plot shows where the robot ends up after walking 10m forward.
- **Blue** dots: Mock backend (ideal, clustered tightly at target)
- **Red** dots: Mock + SensorNoiseModel (simulated real hardware, scattered by slip)
- **Green star**: Target endpoint (10, 0)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.scatter(sim_endpoints[:, 0], sim_endpoints[:, 1], 
           c='steelblue', s=80, alpha=0.7, label='Mock (ideal sim)', zorder=3)
ax.scatter(real_endpoints[:, 0], real_endpoints[:, 1],
           c='indianred', s=80, alpha=0.7, label='Mock + Noise (simulated real)', zorder=3)
ax.scatter([WALK_DISTANCE], [0], c='green', s=200, marker='*', 
           label='Target (10, 0)', zorder=4)

# Draw error ellipses
for endpoints, color, label in [(sim_endpoints, 'steelblue', 'Sim'), 
                                  (real_endpoints, 'indianred', 'Real')]:
    mean = endpoints.mean(axis=0)
    ax.plot(mean[0], mean[1], 'x', color=color, markersize=12, markeredgewidth=2)

ax.set_xlabel('X position (m)', fontsize=12)
ax.set_ylabel('Y position (m)', fontsize=12)
ax.set_title(f'Odometry Drift After {WALK_DISTANCE}m Straight Walk\n'
             f'(Zero-Code Backend Switch: same navigation code, different backends)',
             fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('sim2real_endpoint_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: sim2real_endpoint_scatter.png")

## 5. Transfer Ratio Computation

The **transfer ratio** quantifies how well simulation predictions match reality.
A ratio of 1.0 means perfect transfer; our pass criterion for this test is
`real_error ≤ sim_error × 2.0`.

In [ ]:
from threewe.benchmark.sim2real import (
    STANDARD_TRANSFER_TESTS,
    evaluate_transfer,
)

# Compute endpoint errors
target = np.array([WALK_DISTANCE, 0.0])
sim_errors = np.linalg.norm(sim_endpoints - target, axis=1)
real_errors = np.linalg.norm(real_endpoints - target, axis=1)

sim_mean_error = sim_errors.mean()
real_mean_error = real_errors.mean()

# Use the standard transfer test definition
straight_walk_test = STANDARD_TRANSFER_TESTS[0]  # straight_walk_5m
result = evaluate_transfer(straight_walk_test, sim_mean_error, real_mean_error)

print("=" * 60)
print("TRANSFER VALIDATION: Straight Walk Odometry")
print("=" * 60)
print(f"  Sim endpoint error (mean):  {sim_mean_error:.4f} m")
print(f"  Real endpoint error (mean): {real_mean_error:.4f} m")
print(f"  Transfer ratio:             {result.transfer_ratio:.3f}")
print(f"  Pass criterion:             real ≤ sim × {straight_walk_test.pass_multiplier}")
print(f"  Result:                     {'✅ PASS' if result.passed else '❌ FAIL'}")
print("=" * 60)

## 6. Full Sim2Real Validation Report

Run all 5 standard transfer tests and generate a complete report.
This is the same validation pipeline that the CI runs — but here you can
inspect every intermediate value.

In [ ]:
from threewe.benchmark.sim2real import generate_demo_report

report = asyncio.get_event_loop().run_until_complete(
    generate_demo_report(num_trials=5, seed=42)
)

# Display as Markdown
from IPython.display import Markdown
Markdown(report)

## 7. Noise Parameter Sensitivity

How does odometry drift change as we vary the slip factor?
This sweep helps researchers understand which noise parameters
dominate the sim-to-real gap for their specific task.

In [ ]:
slip_factors = np.linspace(0.0, 0.25, 11)
mean_errors = []

for slip in slip_factors:
    custom_noise = SensorNoiseModel(
        seed=42,
        odometry=noise_model.odometry.__class__(
            slip_factor_min=slip * 0.5,
            slip_factor_max=slip,
            heading_stddev=0.002
        )
    )
    trial_rng = np.random.default_rng(42)
    errors = []
    for sx, sy in sim_endpoints:
        pose = np.array([sx, sy, 0.0], dtype=np.float32)
        noisy = custom_noise.apply_odometry(pose, rng=trial_rng)
        err = np.linalg.norm([noisy[0] - WALK_DISTANCE, noisy[1]])
        errors.append(err)
    mean_errors.append(np.mean(errors))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(slip_factors * 100, mean_errors, 'o-', color='darkblue', linewidth=2)
ax.axhline(y=sim_mean_error * 2.0, color='red', linestyle='--', 
           label=f'Pass threshold (sim×2.0 = {sim_mean_error*2:.3f}m)')
ax.axvspan(5, 15, alpha=0.1, color='orange', label='Hardware-measured range (5-15%)')

ax.set_xlabel('Max Wheel Slip Factor (%)', fontsize=12)
ax.set_ylabel('Mean Endpoint Error (m)', fontsize=12)
ax.set_title('Odometry Drift vs. Wheel Slip Factor\n'
             '(Sensitivity analysis for sim-to-real gap)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sim2real_slip_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: sim2real_slip_sensitivity.png")

## 8. Key Takeaway

This notebook demonstrates the core promise of 3we:

1. **Zero-Code Backend Switch**: The same `Robot` API and navigation code works
   across all backends — only the constructor parameter changes.

2. **Calibrated Noise Models**: The `SensorNoiseModel` is measured from real hardware
   (LD06 LiDAR, BNO055 IMU, mecanum wheels), making simulation predictions
   physically grounded.

3. **Quantified Transfer Gap**: The standard validation protocol provides clear
   pass/fail criteria, enabling researchers to know *before* hardware deployment
   whether their algorithm will transfer.

### Next Steps

- Replace `backend="mock"` with `backend="gazebo"` for physics-based sim
- Add `backend="real"` to compare against actual hardware measurements
- Run `threewe sim2real report` from CLI for automated CI validation
- See `threewe.sim.domain_randomization` for training-time noise injection